In [1]:
import os
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, issparse

c:\Users\Lucifer\anaconda3\envs\rag101\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =============================================================================
# CONFIG
# =============================================================================

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = "example-hybrid-index"
MODEL_NAME = "sentence-transformers/paraphrase-MiniLM-L6-v2"

DOC_FOLDER = Path("./coffee_txt")   # folder containing .txt files

TOP_K = 5

In [3]:
# =============================================================================
# INITIALIZE
# =============================================================================

pc = Pinecone(api_key=PINECONE_API_KEY)

dense_model = SentenceTransformer(MODEL_NAME)

tfidf = TfidfVectorizer(stop_words="english")

In [4]:
# =============================================================================
# DATA LOADING
# =============================================================================

def load_documents(folder: Path) -> List[Dict]:
    docs = []

    for path in sorted(folder.glob("*.txt")):

        try:
            text = path.read_text(encoding="utf-8")
        except:
            text = path.read_text(encoding="latin-1")

        docs.append({
            "id": path.stem,
            "text": text,
            "file_name": path.name
        })

    return docs

In [5]:
# =============================================================================
# SPARSE CONVERSION
# =============================================================================

def csr_to_sparse_dict(row: csr_matrix) -> Dict:
    """Convert CSR row to Pinecone sparse format"""
    return {
        "indices": row.indices.tolist(),
        "values": row.data.astype(np.float32).tolist()
    }

In [6]:
# =============================================================================
# INDEX CREATION
# =============================================================================

def create_index_if_not_exists(dimension: int):

    existing = [i["name"] for i in pc.list_indexes()]

    if INDEX_NAME not in existing:

        pc.create_index(
            name=INDEX_NAME,
            dimension=dimension,
            metric="dotproduct",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )

    return pc.Index(INDEX_NAME)

In [7]:
# =============================================================================
# BUILD VECTORS
# =============================================================================

def build_vectors(docs):

    texts = [d["text"] for d in docs]

    print("Encoding dense vectors...")
    dense_vectors = dense_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    print("Building sparse vectors...")
    sparse_matrix = tfidf.fit_transform(texts)

    pinecone_vectors = []

    for i, doc in enumerate(docs):

        pinecone_vectors.append({
            "id": doc["id"],

            # dense embedding
            "values": dense_vectors[i].tolist(),

            # sparse embedding
            "sparse_values": csr_to_sparse_dict(sparse_matrix[i]),

            "metadata": {
                "file_name": doc["file_name"],
                "text": doc["text"]
            }
        })

    return pinecone_vectors

In [8]:
# =============================================================================
# UPSERT
# =============================================================================

def upsert(index, vectors):

    print(f"Upserting {len(vectors)} documents...")
    index.upsert(vectors=vectors)

In [9]:
# =============================================================================
# QUERY FUNCTIONS
# =============================================================================

def dense_query(index, query: str):

    print("\n=== Dense Search ===")

    q_dense = dense_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    res = index.query(
        vector=q_dense,
        top_k=TOP_K,
        include_metadata=True
    )

    return res

In [10]:
def sparse_query(index, query):

    print("Sparse search")

    q_sparse = tfidf.transform([query])

    dim = dense_model.get_sentence_embedding_dimension()

    zero_dense = [0.0] * dim

    res = index.query(
        vector=zero_dense,
        sparse_vector=csr_to_sparse_dict(q_sparse),
        top_k=TOP_K,
        include_metadata=True
    )

    return res

In [11]:
def hybrid_query(index, query: str, alpha=0.5):

    """
    alpha = weight for dense
    (1-alpha) = weight for sparse
    """

    print(f"\n=== Hybrid Search (alpha={alpha}) ===")

    q_dense = dense_model.encode(
        query,
        normalize_embeddings=True
    )

    q_sparse = tfidf.transform([query])

    res = index.query(

        vector=(q_dense * alpha).tolist(),

        sparse_vector={
            "indices": q_sparse.indices.tolist(),
            "values": (q_sparse.data * (1 - alpha)).tolist()
        },

        top_k=TOP_K,
        include_metadata=True
    )

    return res

In [12]:

# =============================================================================
# PRINT RESULTS
# =============================================================================

def print_results(results):

    for match in results["matches"]:

        print(
            f"{match['id']} | score={match['score']:.4f} | file={match['metadata']['file_name']}"
        )


docs = load_documents(DOC_FOLDER)
print(f"Loaded {len(docs)} documents")

vectors = build_vectors(docs)

index = create_index_if_not_exists(len(vectors[0]["values"]))

upsert(index, vectors)


Loaded 12 documents
Encoding dense vectors...
Building sparse vectors...
Upserting 12 documents...


In [17]:
# =============================================================================
# MAIN
# =============================================================================

def main():

    query = "What is the best way to brew coffee at home?"

    dense_res = dense_query(index, query)
    print_results(dense_res)
    print("=============")

    sparse_res = sparse_query(index, query)
    print_results(sparse_res)
    print("=============")

    hybrid_res = hybrid_query(index, query, alpha=0.1)
    print_results(hybrid_res)
    print("=============")

    hybrid_res = hybrid_query(index, query, alpha=0.5)
    print_results(hybrid_res)
    print("=============")    

    hybrid_res = hybrid_query(index, query, alpha=0.9)
    print_results(hybrid_res)
    print("=============")    


if __name__ == "__main__":
    main()


=== Dense Search ===
cold-brew | score=0.4968 | file=cold-brew.txt
flat-white | score=0.4854 | file=flat-white.txt
french-press | score=0.4603 | file=french-press.txt
espresso | score=0.4530 | file=espresso.txt
latte | score=0.4495 | file=latte.txt
Sparse search
espresso | score=0.1868 | file=espresso.txt
turkish-coffee | score=0.1706 | file=turkish-coffee.txt
flat-white | score=0.0960 | file=flat-white.txt
french-press | score=0.0560 | file=french-press.txt
cold-brew | score=0.0526 | file=cold-brew.txt

=== Hybrid Search (alpha=0.1) ===
espresso | score=0.2134 | file=espresso.txt
turkish-coffee | score=0.1983 | file=turkish-coffee.txt
flat-white | score=0.1350 | file=flat-white.txt
cold-brew | score=0.0970 | file=cold-brew.txt
french-press | score=0.0964 | file=french-press.txt

=== Hybrid Search (alpha=0.5) ===
espresso | score=0.3199 | file=espresso.txt
turkish-coffee | score=0.3094 | file=turkish-coffee.txt
flat-white | score=0.2907 | file=flat-white.txt
cold-brew | score=0.2747 |